# Project 3 - Name Gender Classifier
**Source:** *Natural Language Processing with Python*, exercise 6.10.2

**Goal:** build the best name gender classifier possible, using the Naive Bayes classifier from chapter 6 and incremental feature improvements. Features are tuned on the dev-test set; the test set is used only once at the end.

## 1. Load data and split into train / dev-test / test
The Names Corpus has 7,944 names. The test set gets 500 names, the dev-test set gets 500 names, and the remaining 6,944 names (about the 6,900 in the assignment) form the training set. The names are shuffled with a fixed seed so the result is reproducible.

In [1]:
import nltk, random
from nltk.corpus import names
nltk.download('names', quiet=True)

labeled = ([(n, 'male') for n in names.words('male.txt')] +
           [(n, 'female') for n in names.words('female.txt')])
random.seed(42)
random.shuffle(labeled)

test_names    = labeled[:500]
devtest_names = labeled[500:1000]
train_names   = labeled[1000:]

print('total:', len(labeled))
print('train:', len(train_names), '| dev-test:', len(devtest_names), '| test:', len(test_names))

total: 7944
train: 6944 | dev-test: 500 | test: 500


## 2. Helper function
Trains a Naive Bayes classifier with a given feature function and returns the classifier and its accuracy on the chosen evaluation set.

In [2]:
def evaluate(feature_fn, train, eval_data):
    train_set = [(feature_fn(n), g) for n, g in train]
    eval_set  = [(feature_fn(n), g) for n, g in eval_data]
    clf = nltk.NaiveBayesClassifier.train(train_set)
    return clf, nltk.classify.accuracy(clf, eval_set)

## 3. Incremental improvements (checked on dev-test)
### Version 1 - baseline from the book: last letter

In [3]:
def features_v1(name):
    return {'last_letter': name[-1]}

clf1, acc1 = evaluate(features_v1, train_names, devtest_names)
print('v1 dev-test accuracy:', round(acc1, 3))

v1 dev-test accuracy: 0.754


### Version 2 - add the last two letters and lowercase the name

In [4]:
def features_v2(name):
    name = name.lower()
    return {'last_letter': name[-1],
            'last_two': name[-2:]}

clf2, acc2 = evaluate(features_v2, train_names, devtest_names)
print('v2 dev-test accuracy:', round(acc2, 3))

v2 dev-test accuracy: 0.784


### Version 3 - add the first letter and name length

In [5]:
def features_v3(name):
    name = name.lower()
    return {'last_letter': name[-1],
            'last_two': name[-2:],
            'first_letter': name[0],
            'length': len(name)}

clf3, acc3 = evaluate(features_v3, train_names, devtest_names)
print('v3 dev-test accuracy:', round(acc3, 3))

v3 dev-test accuracy: 0.784


### Error analysis of version 3
Look at the names the classifier gets wrong on dev-test to find patterns for new features.

In [6]:
errors = []
for n, g in devtest_names:
    guess = clf3.classify(features_v3(n))
    if guess != g:
        errors.append((g, guess, n))

print('errors:', len(errors), 'of', len(devtest_names))
for e in sorted(errors)[:30]:
    print('correct=%-7s guess=%-7s name=%s' % e)

errors: 108 of 500
correct=female  guess=male    name=Beatriz
correct=female  guess=male    name=Bliss
correct=female  guess=male    name=Brigid
correct=female  guess=male    name=Brooks
correct=female  guess=male    name=Charil
correct=female  guess=male    name=Dallas
correct=female  guess=male    name=Doralin
correct=female  guess=male    name=Dory
correct=female  guess=male    name=Eden
correct=female  guess=male    name=Eleanor
correct=female  guess=male    name=Evangelin
correct=female  guess=male    name=Gillan
correct=female  guess=male    name=Ginnifer
correct=female  guess=male    name=Glad
correct=female  guess=male    name=Gretchen
correct=female  guess=male    name=Grissel
correct=female  guess=male    name=Harriott
correct=female  guess=male    name=Hatty
correct=female  guess=male    name=Helen-Elizabeth
correct=female  guess=male    name=Hildagard
correct=female  guess=male    name=Ingeberg
correct=female  guess=male    name=Ivett
correct=female  guess=male    name=Ivy


### Version 4 - add the last three letters, first two letters, and a vowel-ending flag

In [7]:
def features_v4(name):
    name = name.lower()
    return {'last_letter': name[-1],
            'last_two': name[-2:],
            'last_three': name[-3:],
            'first_letter': name[0],
            'first_two': name[:2],
            'length': len(name),
            'ends_with_vowel': name[-1] in 'aeiouy'}

clf4, acc4 = evaluate(features_v4, train_names, devtest_names)
print('v4 dev-test accuracy:', round(acc4, 3))

v4 dev-test accuracy: 0.806


### Summary of progress on dev-test

In [8]:
print('%-45s %s' % ('Features', 'dev-test accuracy'))
rows = [('v1: last letter (baseline)', acc1),
        ('v2: + last two letters', acc2),
        ('v3: + first letter, length', acc3),
        ('v4: + last three, first two, vowel ending', acc4)]
for label, acc in rows:
    print('%-45s %.3f' % (label, acc))

Features                                      dev-test accuracy
v1: last letter (baseline)                    0.754
v2: + last two letters                        0.784
v3: + first letter, length                    0.784
v4: + last three, first two, vowel ending     0.806


## 4. Final evaluation on the test set
The final feature set is chosen as the version with the best dev-test accuracy. The test set is evaluated once, with no further changes.

In [9]:
accs = {'v1': (features_v1, acc1), 'v2': (features_v2, acc2),
        'v3': (features_v3, acc3), 'v4': (features_v4, acc4)}
best = max(accs, key=lambda k: accs[k][1])
final_fn, dev_acc = accs[best]

final_clf, _ = evaluate(final_fn, train_names, devtest_names)
test_set = [(final_fn(n), g) for n, g in test_names]
test_acc = nltk.classify.accuracy(final_clf, test_set)

print('Final feature set:', best)
print('dev-test accuracy: %.3f' % dev_acc)
print('test accuracy:     %.3f' % test_acc)
print('difference (dev-test - test): %.3f' % (dev_acc - test_acc))

Final feature set: v4
dev-test accuracy: 0.806
test accuracy:     0.792
difference (dev-test - test): 0.014


## 5. Discussion

In [10]:
import math
se = math.sqrt(test_acc * (1 - test_acc) / len(test_names))
print('Standard error of test accuracy (n=500): about %.3f' % se)

Standard error of test accuracy (n=500): about 0.018


**How does test performance compare to dev-test performance?**
The final classifier (Naive Bayes with the v4 features) reached 0.806 accuracy on the dev-test set and 0.792 on the test set, a drop of about 1.4 percentage points.

**Is this what we'd expect?**
Yes. Two reasons:

1. **The dev-test set was used to make decisions.** Every feature change was kept or dropped based on dev-test accuracy, so the final model is slightly fitted to that particular set of 500 names. The test set was never used during development, so its score is an honest estimate of performance on new names, and it is normally a bit lower.
2. **The sets are small.** With 500 names, the standard error of an accuracy near 0.8 is about 1.8 percentage points. The 1.4-point difference is within that range, so it is consistent with ordinary random variation and does not show serious overfitting.

**Other observations:**
- The biggest gain came from adding the last two letters (v2, +3.0 points). Adding first letter and length (v3) did not help on dev-test, but the suffix and prefix features in v4 did (+2.2 points).
- Many remaining errors are names with no clear gender signal in their spelling (e.g. Eden, Dallas, Brooks, Ivy), so accuracy of about 80% is close to the limit of a simple letter-based feature set.